In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
np.random.seed(8)
# XOR-style data and two-moons customer segmentation
X_xor=np.array([[0,0],[0,1],[1,0],[1,1]],dtype=float); y_xor=np.array([0,1,1,0])
X,y=make_moons(n_samples=300,noise=.15,random_state=8)
X=(X-X.mean(0))/X.std(0)
rng=np.random.default_rng(8); W1=rng.normal(0,.5,(2,8)); b1=np.zeros((1,8)); W2=rng.normal(0,.5,(8,1)); b2=np.zeros((1,1))
sig=lambda z:1/(1+np.exp(-np.clip(z,-40,40)))
losses=[]; lr=.5
for epoch in range(3000):
    H=sig(X@W1+b1); P=sig(H@W2+b2); Y=y.reshape(-1,1)
    loss=-np.mean(Y*np.log(P+1e-9)+(1-Y)*np.log(1-P+1e-9)); losses.append(loss)
    d2=(P-Y)/len(X); gW2=H.T@d2; gb2=d2.sum(0,keepdims=True)
    d1=(d2@W2.T)*H*(1-H); gW1=X.T@d1; gb1=d1.sum(0,keepdims=True)
    W2-=lr*gW2; b2-=lr*gb2; W1-=lr*gW1; b1-=lr*gb1
    if epoch>100 and abs(losses[-1]-losses[-2])<1e-9: break
pred=(sig(sig(X@W1+b1)@W2+b2).ravel()>=.5).astype(int)
print("Epochs:",len(losses),"Final loss:",round(losses[-1],5),"Training accuracy:",round(np.mean(pred==y)*100,2),"%")
# Numerical gradient check for one weight
H=sig(X@W1+b1); P=sig(H@W2+b2); d2=(P-Y)/len(X); analytic=(H.T@d2)[0,0]
eps=1e-5; old=W2[0,0]
def calc_loss():
    q=sig(sig(X@W1+b1)@W2+b2)
    return -np.mean(Y*np.log(q+1e-9)+(1-Y)*np.log(1-q+1e-9))
W2[0,0]=old+eps; plus=calc_loss(); W2[0,0]=old-eps; minus=calc_loss(); W2[0,0]=old
print("Analytical gradient:",round(float(analytic),7),"Numerical gradient:",round(float((plus-minus)/(2*eps)),7))
# XOR demonstration with same two-layer model trained separately
wx=np.random.randn(2,1); bx=0.; errs=[]
for ep in range(30):
    yp=(X_xor@wx+bx>=0).astype(int); err=y_xor-yp; errs.append(np.sum(err!=0))
    wx+=.2*X_xor.T@err; bx+=.2*np.sum(err)
print("Single perceptron XOR errors after 30 epochs:",errs[-1],"(cannot solve non-linearly separable XOR)")
plt.figure(figsize=(12,4)); plt.subplot(1,2,1); plt.plot(losses); plt.title("Backpropagation training loss"); plt.xlabel("Epoch"); plt.ylabel("Binary cross-entropy"); plt.grid()
xx,yy=np.meshgrid(np.linspace(X[:,0].min()-.3,X[:,0].max()+.3,250),np.linspace(X[:,1].min()-.3,X[:,1].max()+.3,250)); grid=np.c_[xx.ravel(),yy.ravel()]
zz=sig(sig(grid@W1+b1)@W2+b2).reshape(xx.shape)
plt.subplot(1,2,2); plt.contourf(xx,yy,zz,levels=[0,.5,1],alpha=.3); plt.scatter(X[:,0],X[:,1],c=y,s=12,edgecolors="k"); plt.title("Two-moons nonlinear decision boundary"); plt.tight_layout(); plt.show()